In [2]:
#installing the xgboost
!pip install xgboost

   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.5/72.0 MB 2.8 MB/s eta 0:00:26
    --------------------------------------- 1.0/72.0 MB 3.0 MB/s eta 0:00:24
    --------------------------------------- 1.6/72.0 MB 3.3 MB/s eta 0:00:22
   - -------------------------------------- 2.1/72.0 MB 2.5 MB/s eta 0:00:28
   - -------------------------------------- 2.9/72.0 MB 2.8 MB/s eta 0:00:25
   -- ------------------------------------- 3.9/72.0 MB 3.2 MB/s eta 0:00:22
   -- ------------------------------------- 4.7/72.0 MB 3.3 MB/s eta 0:00:21
   --- ------------------------------------ 5.5/72.0 MB 3.5 MB/s eta 0:00:20
   --- ------------------------------------ 6.0/72.0 MB 3.3 MB/s eta 0:00:20
   --- ------------------------------------ 6.8/72.0 MB 3.4 MB/s eta 0:00:20
   ---- ----------------------------------- 7.9/72.0 MB 3.5 MB/s eta 0:00:19
   ---- ----------------------------------- 8.7/72.0 MB 3.5 MB/s eta 0:00:19
   ---

In [53]:
#importing all the essential models
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score,classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler
#these models because we want to compare them with xgb
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
#xgb model
from xgboost import XGBClassifier

In [54]:
#reading the file
data = pd.read_csv('Customer_churn.csv')
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [55]:
#dropping the unnecessary column
data.drop('customerID',axis=1,inplace=True)
data.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [56]:
#replacing the empty value with random 0 or 1 value with np.nan method
data.replace(" ",np.nan,inplace=True)
data.dropna(inplace=True)

In [57]:
#turning the whole data from categorical to numerical or one hot encoding
le = LabelEncoder()

for col in data.columns:
    if data[col].dtype == 'object':
        data[col] = le.fit_transform(data[col])

In [58]:
data.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,2504,0
1,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,1465,0
2,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,156,1
3,1,0,0,0,45,0,1,0,2,0,2,2,0,0,1,0,0,42.30,1399,0
4,0,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,924,1


In [59]:
#labelling the dependent and independent variable
y = data['Churn']
X = data.drop('Churn',axis=1)

In [60]:
#splitting the data for training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [61]:
#standardizing the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [62]:
#Logistic regression model
lr = LogisticRegression()
lr.fit(X_train,y_train)
y_lr_pred = lr.predict(X_test)

In [63]:
#Decision Tree model
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train,y_train)
y_dt_pred = dt.predict(X_test)

In [64]:
#Random forest model
rf = RandomForestClassifier(n_estimators=100,random_state=42)
rf.fit(X_train,y_train)
y_rf_pred = rf.predict(X_test)

In [79]:
#XGB model
xgb = XGBClassifier(
    n_estimators= 200,
    learning_rate=0.1,
    subsample=0.1,
    colsample_bytree=0.1,
    max_depth=3,
    eval_metric='logloss')
xgb.fit(X_train,y_train)
y_xgb_pred = xgb.predict(X_test)

In [80]:
#Comparing all the models accuracy
print("Accuracy Score: ",accuracy_score(y_test,y_lr_pred))
print("Accuracy Score: ",accuracy_score(y_test,y_dt_pred))
print("Accuracy Score: ",accuracy_score(y_test,y_rf_pred))
print("Accuracy Score: ",accuracy_score(y_test,y_xgb_pred))

Accuracy Score:  0.7889125799573561
Accuracy Score:  0.7341862117981521
Accuracy Score:  0.7867803837953091
Accuracy Score:  0.7818052594171997


In [ ]:
#from this we can conclude that
#“Logistic Regression outperformed tree-based models, indicating the problem is largely 
#linearly separable and doesn’t require complex feature interactions.”
#and model tree and boost are overfitting 

In [81]:


print(confusion_matrix(y_test, y_xgb_pred))
print(classification_report(y_test, y_xgb_pred))



[[926 107]
 [200 174]]
              precision    recall  f1-score   support

           0       0.82      0.90      0.86      1033
           1       0.62      0.47      0.53       374

    accuracy                           0.78      1407
   macro avg       0.72      0.68      0.69      1407
weighted avg       0.77      0.78      0.77      1407

